# Data Transformation using DSL

### - DSL (Domain Specific Language) ->dropna(),withColumn() etc.
### - Learn how to Read the  data becomes data ingestion developer
### - Learn how to write the  data becomes Data egression developer
### - By Learning Transformation - DATA ENGINEER / DATA ANALYST /ETL DEVELOPER / DATA CURATION DEV


**Transformation we are going to achive in two ways , using DSL approach and using SQL approach**

## DATA Munging (data wrangling)
(Cleanup) Process of transforming and mapping data from Raw form into Tidy(usable) format with the intent of making it more appropriate and valuable for a variety of downstream purposes such for further Transformation/Enrichment, Egress/Outbound, analytics, Datascience/AI application & Reporting

**Types of Munging**
- Passive Data Munging - Data Discovery/Data Exploration/ EDA (Exploratory Data Analytics) (every layers ingestion/transformation/analytics/consumption) - Performing an (Data Exploration) exploratory data analysis of the raw data to identify the attributes and patterns.

- Active Data Munging
    Combining Data + Schema Evolution/Merging/Merging (Structuring)
    Validation, Cleansing, Scrubbing - Cleansing (removal of unwanted datasets), Scrubbing (convert raw to tidy)
    De Duplication and Levels of Standardization () of Data to make it in a usable format (Dataengineers/consumers)

**Difference b/w Active and Passive data munging (from chatGPT)**

**Active Data Munging **- Active data munging involves _**explicit actions performed by the user or developer**_ to clean or transform data.

**Examples:**

- Removing null values
- Converting data types
- Renaming columns
- Filtering rows
- Standardizing date formats
- Aggregating data

**Passive data munging** occurs when **_data preparation happens automatically_** through tools, frameworks, or predefined rules with minimal user intervention.

**Examples:**

- Automatic schema inference when reading CSV files
- Automatic type conversion by ETL tools
- Built-in data quality rules
- Metadata-driven transformations


passive Data Munging - EDA (Exploratory Data Analysis)

**manually understand the Data - manual EDA**
- header
- delimiter
- footer
- columns and datatypes
- comments
- record count
- duplicates / nulls / format issues

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/csv/custsmodified.csv

## Programmatically perform EDA on source data

In [0]:
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/custsmodified.csv",inferSchema=True).toDF("custid","firstname","lastname","age","profession")

cust_df.show(10)
cust_df.printSchema()




In [0]:

print(type(cust_df))
print(cust_df.columns)  #returns column name from CSV

print(cust_df.dtypes)  #returns column name and datatype from CSV
print(cust_df.schema)  #returns schema structure of CSV file so we can assign it in variable and use it if we want but in printSchema (only print the schema)

In [0]:
cust_sample_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/cust_sample.csv")
cust_sample_df.show()

""" 
defaults:
    sep (delimiter)=,
    header=False
    inferSchema=False
"""


In [0]:
cust_sample_schema=cust_df.schema
cust_sample_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/cust_sample.csv",schema=cust_sample_schema)
cust_sample_df.show()

## Distinct() Vs Dropduplicates()

In [0]:
#find number of records in dataframe
print(cust_df.count())


# Deduplication is the process of identifying and removing duplicate records from a dataset to ensure data quality and uniqueness. In PySpark, it is commonly done using distinct() or dropDuplicates().



# check whether DF contains duplicate records  - using distinct - unique records (row level)
#if the entire record is duplicate, we can remove it with distinct and dropDuplicates - both are same


# distinct() -record level duplication - distinct() - checks entire record row by row. if a particular record(row) matches the other row, it considered as duplicate

# dropDuplicates() without argument - record level duplication - similar like distinct
# dropDuplicates() with argument - column level duplication


print(cust_df.distinct().count())     #deduplication on record level

#whereas dropDuplicates() - one option provided by spark, if no arguments passed, it behaves like distinct() function -  checks entire record row by row. if a particular record(row) matches the other row, it considered as duplicate, returning the unique records

print(cust_df.dropDuplicates().count())    #deduplication on record level


#print(cust_df.distinct(["custid"]).count()) - gives error


# find column level unique count

cust_df2 = cust_df.select("custid")

cust_df2.printSchema()
cust_df2.show(5)


print(cust_df.select("custid").distinct().count()) # find column level unique count 
#when we go with select(custid).distinct(), it will return unique custid alone


#case2, need all columns with uniqueness based on custid
#remove duplicates based on custid
print(cust_df.dropDuplicates(["custid"]).count()) # deduplicated on column level and return the entire datframe 




to create a new data frame with deduplicated data

new_df=cust_df.dropDuplicates()

In [0]:
#print unique custid column
#return only custid
cust_df.select("custid").distinct().show(5)

#column level duplication possible only using dropDuplicates()
#remove duplicates based on custid
#return all the column after deduplication based on custid
cust_df.dropDuplicates(["custid"]).show((5))


## Describe() Vs Summary()



**describe()** provides basic descriptive statistics for numeric and string columns.

Statistics Returned
- count
- mean
- stddev
- min
- max

In [0]:
display(cust_df.describe())


**summary()** is more flexible and provides additional statistics.

- count
- mean
- stddev
- min
- 25%
- 50% (median)
- 75%
- max

In [0]:
display(cust_df.summary())

cust_df.summary().show()

In [0]:
cust_df.filter("custid is null").show()

In [0]:
cust_df.select("custid").distinct().show(10)  #it shows only custid columns with deduplicated values

cust_df.dropDuplicates(["custid"]).show(10)   # it returns all columns in df by removing duplicates of custid column

#  scenarios to read files from directory

**suppose i have customer data in diffrent files in same dir - read dir**

**suppose i have customer data in diffrent files in same dir with sub dir as well - read main dir with recursive_lookup enable**


**suppose i have customer data and sales data in diffrent files in same dir , i want to read only sales data - read dir with file pattern (/data/sales*) using pathglobfilter**


** suppose i have customer data and sales data in diffrent files in same dir and sub dir , i want to read only sales data - read main dir with recursivefilelookup and pathGlobfilter="sales*" **

**suppose i have sales data in diff directories - list of path or list of files**

# Schema evolution  - changes in the scehma

Day 1 to Day 5 files have cid ,cname ,age

Day 5 to day 10 file have cid ,cname ,age , profession

Day 11 - cid , cname , mobile , profession

we achived this writing into some columnar parquet / orc file format

while reading the entire data we will use with mergeschema option

combining Data -> schema Evolution / Structuring

# 1. combine data from diffrent files with changes in schema

In [0]:
parquet_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/schema_demo/day3.csv",header=True,inferSchema=True)
parquet_df.show()
parquet_df.printSchema()
parquet_df.write.mode("append").parquet("/Volumes/izwd37dev/wd37db/rawdatta/parquet_write")


In [0]:
parquet_read_df=spark.read.parquet("/Volumes/izwd37dev/wd37db/rawdatta/parquet_write")
parquet_read_df.show()
parquet_read_df.printSchema()

In [0]:
student_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/",header=True,inferSchema=True)
student_df.show()
student_df.printSchema()

In [0]:
student_df1=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part1.csv",header=True,inferSchema=True)
student_df1.show()
student_df1.printSchema()

In [0]:
student_df2=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part2.csv",header=True,inferSchema=True)
student_df2.show()
student_df2.printSchema()



## Merge using union


**SQL union **
 
**union -->** same number of columns and datatypes based on position 

in **sql union --> **return unique records 

in** spark DS - ** union will allow duplicates 

whereas **unionbyName** will work if the columns are in different order. but with same datatype

whereas **unionbyName with allowMissingColumns=True** will merge every all rows and columns and return df with NULL values of missing records






In [0]:

student_df1_df2=student_df1.union(student_df2)
student_df1_df2.show()
student_df1_df2.count()

In [0]:
student_df3=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part3.csv",header=True,inferSchema=True)
student_df3.show()
student_df3.printSchema()


student_df_4=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part4.csv",header=True,inferSchema=True)
student_df_4.show()
student_df_4.printSchema()


# union will work when we have dtafrmes with same number of columns and datatype so it gives error
#error : The value 'chennai' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type

student_df1_df3=student_df1.union(student_df3)
student_df1_df3.show()
student_df1_df3.count()

In [0]:
student_df_5=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part5.csv",header=True,inferSchema=True)
student_df_5.show()
student_df_5.printSchema()

#again number of columns mismatch so it gives error
#error :but the first input has 4 columns and the second input has 3 columns
student_df_5_df1=student_df_5.union(student_df1)
student_df_5_df1.show()
student_df_5_df1.count()


student_df_5_df1_unionbyName_df=student_df_5.unionByName(student_df1,allowMissingColumns=True)
student_df_5_df1_unionbyName_df.show()
student_df_5_df1_unionbyName_df.count()

In [0]:
# SQL union

# union will work on same number of columns and datatypes based on positions
# usually in sql, union will return the unique records

#in spark DS, union will allow duplicates
#unionby name will consolidate all column with allowmissingcolumns=True option


data1=[(100,"raja",23),(102,"ram",25),(103,"ravi",27),(503,"madhu",45)]
data2=[(500,"rajesh",29),(501,"suresh",30),(502,"rajesh",43),(503,"madhu",45)]
data3=[(500,"rajesh",2000),(501,"suresh",2001),(502,"rajesh",2002),(503,"madhu",2003)]
sql_df1=spark.createDataFrame(data1,schema=["id","name","age"])
sql_df2=spark.createDataFrame(data2,schema=["id","name","age"])
sql_df3=spark.createDataFrame(data3,schema=["id","name","year"])

sql_df1.show()
sql_df2.show()

#combine both df into one
combined_df=sql_df1.union(sql_df2)
combined_df.show()
combined_df.count()


print("id, name, age with ID,NAME and Year")
sql_df3.show()
combined_df1=sql_df1.union(sql_df3)
combined_df1.show()
combined_df1.count()

data4=[(100,23,"raja"),(102,25,"ram")]
sql_df4=spark.createDataFrame(data4,schema=["id","age","name"])
sql_df4.show()



#union of id","name","age"with "id","age","name"
print("id, name, age with id,age,name")
combined_df2=sql_df1.unionByName(sql_df4)
combined_df2.show()
combined_df2.count()


#union of id","name","age"with "id","name","year"
print("id, name, age with id,age,name")
combined_df2=sql_df1.unionByName(sql_df3,allowMissingColumns=True)
combined_df2.show()
combined_df2.count()




# 2. Validation, cleansing and scrubbing
handling missing values, handling mull values

**Data munging** (also called **data wrangling**) is the process of transforming and cleaning raw data into a format that can be easily used for analysis, reporting, or machine learning.





In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified


##  2 level cleansing / rejection 

 cleaning up / drop the records 

 null handling - null record removal 

-  null - single column or multiple columns may have null , entire rec may have null 
-  drop.na()


###  clean up data not matching with schema 

###  reject process - 1

**using is not null, is null**



In [0]:

schema_structure="cust_id int,fname string,lname string,age int,profession string"
read_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_structure,mode="PERMISSIVE")
filter_read_df=read_df.filter("cust_id IS NULL OR age IS  NULL")
filter_read_df.show()
filter_read_df.count()










### na.drop() Vs dropna()

na.drop()==dropna()

defaults: subset=all, how=any

In [0]:


# 2 level cleansing / rejection 

# cleaning up / drop the records 

#  null handling - null record removal 

# null - single column or multiple columns may have null , entire rec may have null 

# single null - remove that recod -> delete rec when col is null 
# mulit col null - remove that recod -> delete rec when col is null and col2 is null
# schema_structure="cust_id int,fname string,lname string,age int,profession string"
schema_structure="cust_id int,fname string,lname string,age int,profession string"
read_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_structure,mode="PERMISSIVE")

# handle null -> spark DSL  -> na functions 
# na -> not applicable -> null
read_df.show(10)
print(read_df.count())


# na.drop -> have 3 arg -> subset , how , threshold
# subset -> default is all columns -> option as list of columns 
# how ->default is  any -> options are  ->   all | any 
#               any -> any one column is the subset is null -> remove that record ->  or
#               all -> all columns are null is the subset  -> remove that record -> and




# if all columns are null remove that record 
print("if all columns are null")
null_record_df=read_df.na.drop(how="all")
null_record_df.show(10)
print(null_record_df.count())


print("if any columns have null, remove that record")
# if any one columns have null, remove that record
null_record_df=read_df.na.drop(how="any")
null_record_df.show(10)
print(null_record_df.count())


#custid is the key and it should not have null values so if custid is null, then remove that record
print("if id is the key")
cust_id_null_record_df=read_df.na.drop(subset=["cust_id"])
cust_id_null_record_df.show(10)
print(cust_id_null_record_df.count())



print("custid and age both are null")
#custid and age is the key and it should not have null values so if custid and age both are null, then remove that record
cust_id_null_record_df=read_df.na.drop(subset=["cust_id","age"],how="all")
cust_id_null_record_df.show(10)
print(cust_id_null_record_df.count())


print("custid or age have null")
#custid or age  have null values, then remove that record
cust_id_null_record_df=read_df.na.drop(subset=["cust_id","age"],how="any")
cust_id_null_record_df.show(10)
print(cust_id_null_record_df.count())


In [0]:
#drop - na.drop() or dropna()

schema_structure="cust_id int,fname string,lname string,age int,profession string"
read_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_structure,mode="PERMISSIVE")
read_df.show(10)
print(read_df.count())

#na.drop()=dropna()
#defaults: subset=all, how=any

#using na.drop
#na.drop(subset,how,threshold)
null_dropped=read_df.na.drop()
null_dropped.show(5)
print(null_dropped.count())

#using dropna
null_droppedna=read_df.dropna()
null_droppedna.show(5)
null_droppedna.count()


#threshold -int
#thresh=2 ->atleast 2 columns should not be null
null_dropped_thresh=read_df.na.drop(thresh=2)
null_dropped_thresh.show(5)
print(null_dropped_thresh.count())


#threshold -int
#thresh=2 ->atleast 2 columns should not be null (any 2 columns have column, it will not drop)
null_dropped_thresh=read_df.na.drop(thresh=5)
null_dropped_thresh.show(5)
print(null_dropped_thresh.count())

In [0]:
data=[(1,"James",None,"36636","M",3000),
      (2,"Rose",None,"40288",None,4000),
      (3,None,None,"42114","M",None),
      (4,"Maria","Jones","39192","F",4000),
      (None,"Jen","Mary","Brown",None,-1)
     ]
df=spark.createDataFrame(data,["id","fname","lname","ssn","gender","salary"])
df.show()
df.count()

#default - so any column have null, it will drop that record
drop_df=df.dropna()
drop_df.show()
drop_df.count()


#using all,so if all columns have null, it will drop
drop_df=df.dropna(how="all")
drop_df.show()
drop_df.count()


#using subset - any of these 2 subset have null, it will remove
drop_df=df.dropna(subset=["fname","lname"],how="any")
drop_df.show()
drop_df.count()


#using subset,thresh=1 - any of these 2 subset have value, it will hold the record
drop_df=df.dropna(subset=["fname","lname"],how="any",thresh=1)
drop_df.show()
drop_df.count()


### na.fill()

na.fill -> 3 arg -> subset , value , inplace 

default 0 for numbers and Null for strings

In [0]:
# na.fill -> 3 arg -> subset , value , inplace 
# scrubbing 
# nvl , coalesce 


not_null_df=read_df.na.fill(0,subset=["age"]).na.fill("N/A",subset=["fname","lname"])
not_null_df.show()
not_null_df.count()


In [0]:
# fill - fillna() or na.fill()


data=[(1,"James",None,"36636","M",3000),
      (2,"Rose",None,"40288",None,4000),
      (3,None,None,"42114","M",None),
      (4,"Maria","Jones","39192","F",4000),
      (None,"Jen","Mary","Brown",None,-1)
     ]
df=spark.createDataFrame(data,["id","fname","lname","ssn","gender","salary"])

df.show()
df.printSchema()
print(df.count())



# na.fill()= fillna()
# for integer,decimal 0 and string "NA"
# date and timestamp column, fill will not work

#assign/ fill 0 value to all integer column which have NULL
nonull_df=df.na.fill(0)
nonull_df.show()

#assign/ fill 0 value to specific id column which have NULL
df.na.fill(0,subset="id").show()



#assign/ fill 0 value to all integer column which have NULL
df.na.fill(0,subset="id").fillna("unknown",subset="fname").show()


#assign/ fill 0 value to all integer column which have NULL
df.na.fill(0,subset="id").fillna("unknown",subset=["fname","lname"]).show()


#assing 0 for integer and unknown for string datatype which have NULL
df.na.fill(0).na.fill("NA").show()

### replace()

In [0]:
#replace - it will replace everywhere in the dataset by default
#if you want to limit the replace to particular column, use subset
#to specify more values, use dictionary or list
#replace (oldvalue,newvalue,subset)
df.show()
df.count()

#for one value 
replace_df=df.replace("James","John")
replace_df.show()

#for multiple values use dictionary,list
replace_df=df.replace({"Jones":"arjun","Rose":"Mary"})
replace_df.show()

#for multiple values use lsit
replace_df=df.replace(["M","F"],["Male","Female"])
replace_df.show()


#for multiple values use dictionary
replace_df=df.replace({3000:30000,-1:50000,4000:40000},subset="salary")
replace_df.show()

### filter() and select()

filter() - similar to where condition. when you want to show a particular column based on certain condition

select() - used to show some specific column

In [0]:
schema_structure="id int,fname string,lname string,age int,profession string,corrupt_record string"
read_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_structure,mode="PERMISSIVE",columnNameOfCorruptRecord="corrupt_record")
read_df.show(5)
read_df.printSchema()

id_df=read_df.select("id")
id_df.show(5)

error_df=read_df.filter("corrupt_record is not null")
error_df.show(10)  # 5 records

error_df=read_df.filter("corrupt_record is not null")
error_df.show(10)  # 5 records

valid_df=read_df.filter("corrupt_record is null")
valid_df.show(10)  #10000 records


#error_df.cache()
error_df.select("corrupt_record").show()


#how to write the error record into separate table
error_df.write.mode("overwrite").csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/error_csv")




# 3. Standarisation

**Making the data more standard by adding,removing,reordering column as per the expected standard, unifying into expected format, converting the type as expected etc**

###  Standarisation 1 - adding audit columns


### i. using withColumn- col(),lit() and current_date()

.withColumn("columnname",value)

In [0]:

# additional audit columns 
# load_date, system ,mod_date, user,applicatioid

# how to add columns with staic value in pyspark  
#   -- using withColumn and lit()

# SQL -> select 'customer' as system,current_date as load_date , * from staging_cust

# DSL (domain specific language) --> select , withColumn
# withColumn(new_col_name,value ) 
# value -> should be in the column format
# select *,'customerdata' as source from tbl

# withcolumn -> hardcoded using lit 
#            -> taking another column using col
#            -> using built in sql function which return column

read_df.columns

#lit = create a value for the new column 
#col = if already a column in dataframe, just want to create new column and copy the values from existing column
from pyspark.sql.functions import lit,col,current_date

#lit = create a value for the new column 
#col = if already a column in dataframe, just want to create new column and copy the values from existing column
from pyspark.sql.functions import lit,col
#how to add a column to datasource using withColumn in DSL


#ERROR: Argument `col` should be a Column, got str.
#read_enrichment_df=read_df.withColumn("Source","Customerdata")  #- error because withcolumn value cannot be string


#for new value
read_enrichment_df=read_df.withColumn("Source",lit("Customerdata"))
read_enrichment_df.show(5)

#for existing value in df and copy the value as a new column
read_enrichment_df=read_df.withColumn("Dupli_Profession",col("profession"))
read_enrichment_df.show(5)


#for doing multiple columns
read_enrichment_df=read_df.withColumn("Source",lit("Customerdata")).withColumn("Job_id",lit(101)).withColumn("Prof",col("profession")) 
read_enrichment_df.show()
read_enrichment_df.printSchema()

#to load currentDate() as new column
read_enrichment_df=read_df.withColumn("Source",lit("Customerdata")).withColumn("Job_id",lit(101)).withColumn("Prof",col("profession")).withColumn("Load_Date",current_date())
read_enrichment_df.show()
read_enrichment_df.printSchema()


#how to add a column to datasource using withColumn in DSL


#ERROR: Argument `col` should be a Column, got str.
#read_enrichment_df=read_df.withColumn("Source","Customerdata")  #- error because withcolumn value cannot be string


#for new value
read_enrichment_df=read_df.withColumn("Source",lit("Customerdata"))
read_enrichment_df.show(5)

#for existing value in df and copy the value as a new column
read_enrichment_df=read_df.withColumn("Dupli_Profession",col("profession"))
read_enrichment_df.show(5)


#for doing multiple columns
read_enrichment_df=read_df.withColumn("Source",lit("Customerdata")).withColumn("Job_id",lit(101)).withColumn("Prof",col("profession")) 
read_enrichment_df.show()
read_enrichment_df.printSchema()

#to load currentDate() as new column
read_enrichment_df=read_df.withColumn("Source",lit("Customerdata")).withColumn("Job_id",lit(101)).withColumn("Prof",col("profession")).withColumn("Load_Date",current_date())
read_enrichment_df.show()
read_enrichment_df.printSchema()



replace("NULL","PILOT",subset=["profession"]) will work the column contains the string "NULL" (as text)

if the column contains actual null values (displayed as null in .show()), replace() will not work. In that case, use:

df = df.fillna({"profession": "Pilot"})

In [0]:
#profesion wise count using groupby

read_profession_df=read_enrichment_df.groupBy("profession").count()
read_profession_df.show(50,False)


#NULL have 90 records
#Pilot has 210 records

#i want to update NULL to Pilot
read_profession_df.filter(col("profession").isNull()).show()
update_pilot_df=read_profession_df.na.fill("PILOT",subset=["profession"])
update_pilot_df.show(100,False)

update_pilot_df.groupBy("profession").count().show(50)



In [0]:
# read csv  -> Df 
 
# default -> all cols string  ,
# inferschema -> true -> based on your data schema will defined automatically

# schema - structtype -> structfield or ddl string ( recomannded for large data )

# rejection rule -default 3 options (mode) spark provoiding while loading data -> permissive / dropmalformed / failfast 

# permissive - allow everything regardless of schema -> null for wrong data
# dropmalformed - drop the corrupted data 
# failfast - throw error if any data is wrong

# RCA om the bad record / collect the bad record to correct later
# permissive + columnNameOfCorruptedRecord -> add new column to the dataframe (error_rec string)

In [0]:

from pyspark.sql.functions import lit, col, current_date

df=spark.range(20)
df.show(5)
df.printSchema()
print(df.schema)


#create 3 more columns
#create new column called id2= id column *2 (using col)
#Create load_dt as current_date (using builtin funciton)
#create createdby as dbuser (using lit)
#equivalent sql - select id2 as id*2, current_date() as load_dt, lit("dbuser") as createdby

df2=df.withColumn("id2",col("id")*2).withColumn("load_dt",current_date()).withColumn("createdby",lit("dbuser"))
df2.show(5)
df2.printSchema()



import getpass
current_user = getpass.getuser()
#create new column as current_user

print("Updated dataframe")
df3=df2.withColumn("CurrentUser",lit(current_user))
df3.show(5,False)





In [0]:
data=[(1),(2)]

data_df=spark.createDataFrame(data,["emp_id"])
data_df.show()
data_df.printSchema()

### ii. using select

.select(value.alias("column_name"))

In [0]:
#using withcolumn we added columns
#now using select

from pyspark.sql.functions import *
df=spark.range(20)
df.show(5)
df.printSchema()
print(df.schema)

df2=df.select("id",(col("id")*2).alias("id2"),current_date().alias("load_dt"),lit("dbuser").alias("createdby"))
df2.show(5)
df2.printSchema()

#using select we can add columns
#using withcolumn we can add columns and rename columns

df2.show(5)
df2.printSchema()





In [0]:
schema_structure="cust_id int,fname string,lname string,age int,profession string"
read_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_structure,mode="PERMISSIVE")
read_df.show(5)
read_df.select("*").show(5)
read_df.select("fname","lname").show(5)
read_df2=read_df.withColumn("fullname",concat(col("fname"),lit(" "),col("lname")))
read_df2.show(5)

#using select
read_df3=read_df.select("fname","lname",concat(col("fname"),lit(" "),col("lname")).alias("fullname"))
read_df3.show(5)
read_df3.printSchema()




## Standarization -2 uniformality

In [0]:
read_df.show(5)

from pyspark.sql.functions import upper
#profession, receiving in multiple cases, lower or upper,initcap

standard_df=read_df.withColumn("profession",upper(col("profession")))
standard_df.show(5)
standard_df.printSchema()

standard_df=read_df.withColumn("Updated_Prof",upper(col("profession")))
standard_df.show(5)
standard_df.printSchema()

standard_df=read_df.select(upper(col("profession").alias("Updated_Prof")))
standard_df.show(5)
standard_df.printSchema()


standard_df2=read_df.select("profession",upper(col("profession")).alias("Updated_Prof"))
standard_df2.show(5)
standard_df2.printSchema()

standard_df3=read_df.select("profession",upper(col("profession")).alias("Updated_Prof"),concat(col("fname"),lit(" "),col("lname")).alias("FullName"))
standard_df3.show(5)
standard_df3.printSchema()





# Standarisation 3 - typecasting

using .cast("int")

 

### i. regular expression - rlike

In [0]:
read_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",mode="PERMISSIVE").toDF("cust_id","fname","lname","age","profession")
read_df.show(5)
read_df.printSchema()

#sql - select cast(age as int) as age from cust_df

from pyspark.sql.functions import col
#filter out the number that causes the error and do the type conversion
read_df2=read_df.filter("age!='7-7'").withColumn("age",col("age").cast("int"))
read_df2.show(5)
read_df2.printSchema()


#using regular expression for non standard record (to get integer - ^[0-9]+$)

read_df.filter("age rlike '^[0-9]+$'").show()
read_df.filter("age not rlike '^[0-9]+$'").show()

read_df2=read_df.filter("age rlike '^[0-9]+$'").withColumn("age",col("age").cast("int"))
read_df2.show(5)
read_df2.printSchema()



## Standarisation - 4 - Naming and Reordering

### i. withColumnRenamed - use to rename the column

## ii. for select use alias to rename the column 

In [0]:
read_df.show(5)

 #rename the column

#using withColumnRenamed options
read_df2=read_df.withColumnRenamed("fname","firstName").withColumnRenamed("lname","lastName")
read_df2.show(5)
read_df2.printSchema()


#using select option
read_df2=read_df.select("cust_id",col("fname").alias("firstName"),col("lname").alias("lastName"),"age","profession")
read_df2.show(5)


### iii. drop() - to delete the column from df

### iv. using select() - Rearrange the column 

In [0]:
#remove a column
#select only required column
#drop
from pyspark.sql.functions import *


#using withcolumn
read_df1=read_df.withColumn("fullname",concat(col("fname"),lit(" "),col("lname")))
read_df1.show(5)
read_df1.printSchema()

#now i want to drop the fname and lname column
read_df2=read_df1.drop("fname","lname")
read_df2.show(5)

#using select rearranging the order i want - custid,name,age,prof
read_df3=read_df2.select("cust_id","fullname","age","profession")
read_df3.show(5)



In [0]:

#input_file_name is a spark fnction but not working in databricks but it so use _metadata

filename = "cust_info_south_20260618.csv"

base = filename.replace(".csv", "")
parts = base.split("_")

print(parts)
source = parts[-2]
data_dt = parts[-1]

print(source)   # south
print(data_dt)

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/cust_data/emp_NAC_20260619.csv

# Usecase 
we recivied data from multiple sources (regions) we have to load data into our storage

- csv with header
- file name : empdata_region_datadt.csv
- cid,fname,lname,age,prof

output :

cid,fullname,age,prof,load_dt,data_dt,source,created_by

mappings :

- cid->cid
- fullname -> fname+lname
- age->age
- prof ->prof
- load_dt -> current date
- data_dt -> take from the file name
- source -> take from the file name
- created by -> current user

In [0]:
#input_file_name is a spark fnction but not working in databricks but it so use _metadata

from pyspark.sql.functions import input_file_name,col,lit,split,current_user,concat

#read csv file

cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/cust_data").toDF("cust_id","fname","lname","age","profession")



#to get the file name from directory data using(input_file_name but its not working in databricks so we use _metadata)

#Error : The command(s): input_file_name are not supported in Unity Catalog. Please use _metadata.file_path instead. SQLSTATE: 0AKUC
cust_df_metadata=cust_df.withColumn("filename",col("_metadata"))
cust_df_metadata.printSchema()

#taking filename alone from metadata
cust_df_filename=cust_df_metadata.withColumn("filename",col("_metadata.file_name"))
cust_df_filename.show(5,False)

#removing .csv from filename using split option
cust_df_filename=cust_df_metadata.withColumn("filename",split(col("_metadata.file_name"),"\\.")[0])
cust_df_filename.show(5,False)

#split the filename for source and datadate column
cust_df_source_datadt_df=cust_df_filename.withColumn("source",split(col("filename"),"_")[1]).withColumn("data_dt",split(col("filename"),"_")[2]).drop("filename")


#adding createdby Column with createduser
cust_df_createduser=cust_df_source_datadt_df.withColumn("createdby",current_user())


#add fname and lname and make it a fullname
cust_df_fullname=cust_df_createduser.withColumn("fullname",concat(col("fname"),lit(" "),col("lname"))).drop("fname","lname")

cust_df_final=cust_df_fullname.select("cust_id","fullname","age","profession","source","data_dt","createdby")
cust_df_final.show(10)
cust_df_final.printSchema()

cust_df_final.write.mode("overwrite").saveAsTable("izwd37dev.wd37db.cust_info_table")


#cust_df_source_datadt_df.write.mode("overwrite").saveAsTable("cust_info_silver")









In [0]:
%sql

select(*) from izwd37dev.wd37db.cust_info_table

# Select Vs SelectExpr

select - is for pyspark dsl col function

selectExpr - for pyspark sql expression



## _metadata()

In [0]:
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/cust_data",header=True,inferSchema=True)

cust_df.select(col("_metadata")).printSchema()

## withColumn Vs withColumns

In [0]:
#withColumn - adding one column
#withColumns - adding multiple columns using dictionary (key,value pairs)



# Data Enrichment - Detailing of data
Makes your data rich and detailed

a. Add (withColumn,select,selectExpr), Derive (withColumn,select,selectExpr), Remove/Eliminate (drop,select,selectExpr), Rename (withColumnRenamed,select,selectExpr), Modify/replace (withColumn, select/selectExpr) - (very important spark sql functions)

b. split, merge/Concat

c. Type Casting, reformat & Schema Migration


# Data Customizations

user Defined functions (UDF) using def

upper , lower , concat ,lit , initcap, col ,current_date ... -> built in functions

how to use your python function inside the select / withcolumn

## i. create udf using python function (def)

In [0]:
# 1. import udf -> this will convert python fnction into spark udf
from pyspark.sql.functions import udf

# 2. create your python function using def | lambda ...
def wd37_custom_upper(strValue):
    return strValue.upper()+"$"+strValue.lower()

# 3. convert python function to udf 
#  udf will take function and return type as input , default retrun string type
udf_upper=udf(wd37_custom_upper)

print(wd37_custom_upper("databricks spark"))

# 4 we can use converted udf into  pyspark select / withcolumn 

cust_df.select("*",udf_upper(col("profession"))).show(10,False)

## ii. create udf using anonymous (lambda) function

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
udf_test=udf(lambda a:a*a,IntegerType())

data=spark.range(10)
data.show()
data_df=data.select("id",udf_test(col("id")).alias("id2"))
data_df.show()
data_df.printSchema()


# 2. Derive the flag/indicator from the existing column 

In [0]:
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/custs_header",header=True,inferSchema=True)
cust_df.show(10)

#categorise age into multiple category

#age < 18 is minor
#age >18 and <55 - midage
#age>55 is senior citizen

#create udf using python function (def)
from pyspark.sql.functions import udf,col,lit
from pyspark.sql.types import StringType


def get_age_calc(age):
  if age < 18:
    return "minor"
  elif age > 18 and age <= 55:
    return "midage"
  elif age > 55:
    return "senior"

print(get_age_calc(56))
print(get_age_calc(10))


#udf conversion
#udf takes function as a input
#return type - default: string type()
# if you want other type, need to mention   integerType()


udf_get_age_calc=udf(get_age_calc,StringType())    #sending function as a input

cust_df_age=cust_df.withColumn("age_group",udf_get_age_calc(col("age")))
cust_df_age.show(10)
cust_df_age.printSchema()




In [0]:
cust_df.show(10)

from pyspark.sql.functions import udf,col,lit
cust_df_age=udf(lambda age: "minor" if age < 18 else "midage" if age > 18 and age <= 55 else "senior")(col("age"))
cust_df_age=cust_df.select("*",cust_df_age.alias("age_group"))
cust_df_age.show(10)
cust_df_age.printSchema()



# 3. case when 







### i. using .when and .otherwise

case when using DSL function 

- when(condition1,value_1).otherwise(value_2)
- when(condition1,value_1).when(condition2,value_2).otherwise(value_default)

In [0]:
%python
#case when if age<18 minor then when age>18 and age<=55 midage else senior end

#when(condition,value_1).otherwise(value_2)
#when(age<18,"minor")

from pyspark.sql.functions import when

cust_age_case_when_dsl_df=cust_df.withColumn("age_group",when(col("age")<18,"Minor"))
cust_age_case_when_dsl_df.show(10)
cust_age_case_when_dsl_df.printSchema()

cust_age_case_when_dsl_df=cust_df.withColumn("age_group",
                                             when(col("age")<18,"Minor")
                                             .when((col("age")>=18) & (col("age")<=55),"Midage")
                                             .otherwise("Senior"))
cust_age_case_when_dsl_df.show(10)
cust_age_case_when_dsl_df.printSchema()


### ii. using selectExpr

In [0]:
cust_age_select_dsl_df=cust_df.selectExpr("*","case when age < 18 then 'Minor' when age >= 18 and age <= 55 then 'Midage' else 'Senior' end as age_group")
cust_age_select_dsl_df.show(10)
cust_age_select_dsl_df.printSchema()


In [0]:
# case when age <18 then 'minor' when age <55 then 'midage' else 'senior' end 

# select -> col , dsl function 
# selectExpr -> col, sql expression
# df.select(current_date(),expr("concat('a','b')"))
# expr is the dsl fucntion it will take sql expression as input return a column 

from pyspark.sql.functions import expr

cust_df.withColumn("age_cat",expr("case when age <18 then 'minor' when age <55 then 'midage' else 'senior' end ")).show()

In [0]:
sample_df=spark.range(10)

from pyspark.sql.functions import expr
sample_df.show()
sample_df2=sample_df.select("id",expr("id * id as id2"), expr("'test' as test"))
sample_df2.show()

sample_df2.where(expr("id<5")).show()

# 3. Date Formating

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/json/emp1.json

In [0]:
# date function 
# default date format - yyyy-MM-dd
# convert string  to date - to_date(string,format)
# convert date to string format - date_format(date,format)

from pyspark.sql.types import *
from pyspark.sql.functions import to_date,date_format


source_schema= StructType([
    StructField("branch_id",IntegerType(),True),
    StructField("emp_id",IntegerType(),True),
    StructField("firstname",StringType(),True),
    StructField("lastname",StringType(),True),
    StructField("profession",StringType(),True),
    StructField("age",IntegerType(),True),
    StructField("joining_date",StringType(),True),
    StructField("location",StringType(),True)
])

json_df=spark.read.json("/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json",multiLine=True,schema=source_schema)
json_df.show()
json_df.printSchema()

# convert joining date into a date 

# date -> dd/MM/yyyy

emp_df1=json_df.withColumn("date_join",to_date(col("joining_date"),"dd/MM/yyyy"))

emp_df1.show()
emp_df1.printSchema()

# date -> dd/MM/yyyy into "yyyy-MM-dd"
emp_df2=emp_df1.withColumn("str_date",date_format(col("date_join"),"yyyy-MMM-dd"))

emp_df2.show()
emp_df2.printSchema()



# sending to downstream they want date MMM/dd/yyyy

emp_df3=emp_df2.withColumn("str_date",date_format(col("date_join"),"MMM/dd/yyyy"))

emp_df3.show()
emp_df3.printSchema()

# Data core curation (Pre Wrangling)

- select,
- filter, 
- format,
- Group and Agg, 
- sorting

## Select

In [0]:
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/custs_header",header=True,inferSchema=True)
cust_df.show(5)

from pyspark.sql.functions import col,when,expr
#select only few columns - cust_id,fname,age

cust_selected_df=cust_df.select("custid",col("fname"),cust_df.age)
cust_selected_df.show(5)


#is_vote =yes or no - deriving a column from existing col

#using selectExpr
print("using SelectExpr")
cust_vote_df=cust_df.selectExpr("*","case when age>=18 then 'Yes' else 'No' end as is_vote")
cust_vote_df.show(5)

#using pure DSL
print("using withColumn, when and otherwise")
cust_vote_withCol_df=cust_df.withColumn("is_vote",when(col("age")>=18,"Yes").otherwise("No"))
cust_vote_withCol_df.show(5)

print("using withcolumn and expr")
cust_vote_withCol_expr_df=cust_df.withColumn("is_vote",expr("case when age>=18 then 'Yes' else 'No' end as is_vote"))
cust_vote_withCol_expr_df.show(5)


#implement multiple conditions using &
cust_vote_multiple_cond_df=cust_df.withColumn("is_vote",when((col("age")>=18) & (col("profession")=="Pilot"), 'Yes').otherwise('No'))
cust_vote_multiple_cond_df.show(5)






## filter

In [0]:
# we can use filter or where
cust_filter_df=cust_vote_df.filter("is_vote='No'").show(5)


cust_filter_df=cust_vote_df.where("is_vote='Yes'").show(5)
cust_filter_df=cust_vote_df.where("is_vote='Yes' and profession='Pilot'").show(5)

## Format, cast

In [0]:
#add new column dt and format it into yyyyMMdd
#to_Date convert string to date -- >output will be date type
#date_format convert date to string -->output will be string
#cast used to change the datatype


from pyspark.sql.functions import current_date,date_format
cust_final_df=cust_vote_df.withColumn("date",current_date())
cust_final_df.show(5)

#now i have date, but want to convert into string format

cust_final_df1=cust_final_df.withColumn("date(yyyyMMdd)",date_format(col("date"),"yyyyMMdd"))
cust_final_df1.show(5)
cust_final_df1.printSchema()


# convert string format date to int format
cust_final_df1=cust_final_df.withColumn("date(yyyyMMdd)",date_format(col("date"),"yyyyMMdd").cast("int"))
cust_final_df1.show(5)
cust_final_df1.printSchema()
,

## Group and Aggregations

- groupby - Grouping column, dimension data eg:city wise,state wise
- Aggregations - measures, mostly with numbers,sales, salary, count, avg, sum, min, max
- orderby (ascending,descending) for sorting

In [0]:
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/custs_header",header=True,inferSchema=True)
cust_df.show(5)

#Group by professsion wise 
#groupBy if we print the output, it will be a grouped object, so show, print,display cannot be used.
#so groupBy used with aggregated functions.
group_prof_df=cust_df.groupBy("profession")
print(group_prof_df)  #GroupedData[grouping expressions: [profession], value: [custid: int, fname: string, lname: string, age: int, profession: string], type: GroupBy]

#aggregated functions - max,min,count
#Group by professsion wise count
group_prof_df=cust_df.groupBy("profession").count()
group_prof_df.show(5)


#Groupby profession wise max age
group_prof_df=cust_df.groupBy("profession").max("age")
group_prof_df.show(5)

#Groupby profession wise min age
group_prof_df=cust_df.groupBy("profession").min("age")
group_prof_df.show(5)

#group_prof_df=cust_df.groupBy("profession").min(cust_df.age)   #AssertionError: Column age not found
#group_prof_df.show(5)

#groupby both age and profesion
group_age_prof_df=cust_df.groupBy("profession","age").count()
group_age_prof_df.show(5)

group_age_prof_df=cust_df.groupBy("profession","age").count().orderBy("age")
group_age_prof_df.show()

cust_df.groupBy("age","profession").count().orderBy("age",asceding=True).show(5)
cust_df.groupBy("age", "profession").count().orderBy("age", ascending=False).show(5)

cust_df.groupBy("age","profession").count().orderBy(["age","count"],asceding=[False,True]).show()


In [0]:

from pyspark.sql.functions import count,max,min
#profession wise min age, max age, count of records
#after groupby only we can use agg
#if we are doing single aggregation no need to gofor agg -- .groupBy("profession").count()
#when do multiple aggregation - use .groupBy("profession").agg(count("*"),min("age"),max("age"))


group_prof_df=cust_df.groupBy("profession").agg(count("*"),min("age"),max("age"))
group_prof_df.show(5)

group_prof_df=cust_df.groupBy("profession").agg(min("age"),max("age"),count("*"))
group_prof_df.show(5)

#adding column name for each aggregation
group_prof_df=cust_df.groupBy("profession").agg(count("profession").alias("count"),min("age").alias("min_age"),max("age").alias("max_age"))
group_prof_df.show(5)

cust_df.filter("profession='Veterinarian'").show(5)
cust_df.selectExpr("profession='Pilot'").show()


#select min(age) from table
cust_df.selectExpr("min(age) as age").show()

# select custif,fname,lname ,age from tbl

# select age , count(1) from tbl group by age -> age,count

# order by - 

cust_df.groupBy("age","profession").count().orderBy("age",ascending=False).show()

cust_df.groupBy("age","profession").count().filter("age=21").orderBy(["age","count"],ascending=[False,True]).show(100)




In [0]:
# df.groupby("profession").count()  - returns df
# df.groupby("profession").agg(count()) -- returns df

from pyspark.sql.functions import count,max,min

cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/custs_header",header=True,inferSchema=True)
cust_df.show(5)

#sorting the records based on age in descending order. in spark descending=True will not work so use ascending =False
agg_df=cust_df.orderBy("age",ascending=False)
agg_df.show()

agg_df1=cust_df.groupBy("age").count()
agg_df1.show()

#find the last age 

agg_df1.orderBy("age",ascending=False).show()

#find the prof wise count, min age,max age
agg_df2=cust_df.groupBy("profession").agg(count("*").alias("TotalCount"),min("age").alias("min_age"),max("age").alias("max_age"))
agg_df2.show()





### parse the customer data and add new column age_cat
 
  --->Young (<=25)
  --->mid_age (<25 and >55)
  --->senior (above 55)

  total_rec,min_age,max_age

In [0]:
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/custs_header",header=True,inferSchema=True)
cust_df.show(5)

from pyspark.sql.functions import expr
df=cust_df.withColumn("age_category",expr("case when age<=25 then 'young' when age>25 and age>55 then 'middle' else 'senior' end"))
df.show()

final_df=df.groupBy("age_category").agg(count("age_category").alias("total_count"),min("age").alias("min_age"),max("age").alias("max_age"))
final_df.show()


#order the results based on age category

final_df.orderBy("age_category",ascending=True).show()


# show(),limit()and take()
Difference between show(),limit()and take()

anything returns other than DF is a action

- show()- just showing the records and returns nothing  ----> return type : nothing  -- action

- limit() - returning a DF, from sourcce df  ----> return type : DF   -- transformation

- take() - return the limited record in python object which is list ----> return type : Python object -- action

In [0]:

df.limit(10)   #its a transformation and returns DataFrame[custid: int, fname: string, lname: string, age: int, profession: string]
df.take(10)    #its a action -- returns list of records in tuples immediately 

#take 10 records from cust_df and store in another df

df_limit=cust_df.limit(10)
df_limit.show()
df_limit.count()

#we can do with take also
df_take=cust_df.take(10)
print(df_take)    # returns list of tuples
display(df_take)


# Data Wrangling

-- Joins  - combining multiple tables based on some conditions or no conditions

-- union also helps to join the tables

## join and Union difference


**join vs union**

**join - horizontal - adding columns - enriching with columns fron the other table - widenining**

**union - vertical - adding rows - making a table taller**


union combines the table1 with table2 vertically ( adding more rows) and makes the table1 taller -- like adding a rows /records

unionbyName allow missing columns -add more rows and making missing column value null

Join combines the table 1 with table2 horizontal (widening the table) -- like adding a columns

## Joins
-  left join
-  right join
-  inner join
-  full join  - combine all records
-  self join  - joining with the same table (any parent child relationship, we go with self)
-  cross join (m*n)
-  semi join
- anti join

heavily use left join and inner join

**default join in inner join in spark**


Spark internals - when the data shuffle happened,this magic happen in spark internally.(performance side)
Broadcast join
Shuffle join
sort merge join
AQE - Adaptive Query Execution

### Left Join

In [0]:
#Left join / left outer - it will bring all records from left and bring matching records from the right table
#Left join syntax
#df1.join(df2,on,how)

#df1.join(df2,condition,join_type)
#df1.join(df2,on,how)

#when joining 3 tables - use the below syntax
#df1.join(df2,on,how).join(df3,on,how)

emp_data=[(100,"A0",25),(101,"A1",36),(102,"A2",26),(103,"A3",32),(104,"A4",12),(105,"A5",54),(106,"A6",32)]

city_data=[(100,"chn"),(104,"blr"),(105,"hyd"),(106,"mum"),(107,"hyd"),(110,"del")]

emp_data_df=spark.createDataFrame(emp_data,["eid","ename","age"])
city_data_df=spark.createDataFrame(city_data,["eid","city"])

emp_data_df.show()
city_data_df.show()



print("Left Join - return all columns from left and matching id columns from right")
#join column name is same on both df, simply we give column name in the condition
emp_city_left_df=emp_data_df.join(city_data_df,["eid"],"left")

print("Left Join - return all records from emp and matching city records from city")
emp_city_left_df.show()


  


In [0]:
#case 2:
#in sql, join can be used in two ways - "on", "using"
#select * from emp left join city on emp.eid=city.eid
# id, name,age,id,city 


# on returns dataframe with 2 id columns
# using returns dataframe with one id column

# eid,ename,age,eid,city - sql output (in result, two eid column returns)
emp_city_left_df=emp_data_df.join(city_data_df,on=emp_data_df.eid==city_data_df.eid,how="left")
print("Left Join - return all records from emp and matching city records from city")
emp_city_left_df.show()


#select * from emp left join city using(eid)    

In [0]:
#case-3:
#join columns with different name

#emp_df= eid,name,age
#city_df= id,city
# join condition -> id=eid



# col() -> existing column 
# df.col_name -> columnn
# lit() -> literal value / hard coded 

from pyspark.sql.functions import col,expr
city_df_2=city_data_df.withColumnRenamed("eid","id")
city_df_2.show()

emp_city_df=emp_data_df.join(city_df_2,on=emp_data_df.eid==city_df_2.id,how="left")
emp_city_df.show()

emp_city_df=emp_data_df.join(city_df_2,on=col("eid")==col("id"),how="left")
emp_city_df.show()


emp_city_df=emp_data_df.join(city_df_2,on=expr("eid=id"),how="left")
emp_city_df.show()

In [0]:
# case 4 :
# if column names same in both dataframe - ambiqiuos 

# table alias 

city_df_2=city_data_df.withColumnRenamed("id","eid")
city_df_2.show()


#Reference `eid` is ambiguous, could be: [`eid`, `eid`]. SQLSTATE: 42704
emp_city_df=emp_data_df.join(city_df_2,on=col("eid")==col("eid"),how="left")

emp_city_df.show() # gives ambiguous error and it can be solved using alias method

emp_city_df=emp_data_df.alias("e").join(city_df_2.alias("c"),on=expr("e.eid=c.eid"),how="left") 
emp_city_df.select("e.*","c.city").show()

### Right Join / right outer

In [0]:
#right join  - it will bring all records from right and bring matching records from the left table

emp_data_df.show()
city_data_df.show()

print("Right Join - return all columns from right and matching id columns from left")
emp_city_right_df=emp_data_df.join(city_data_df,["eid"],"right")
emp_city_right_df.show()


### Inner Join / equi join

In [0]:
#inner join - it will bring only matching records from both the tables

emp_data_df.show()
city_data_df.show()

print("Inner Join - return only matching id records from both table")
emp_city_inner_df=emp_data_df.join(city_data_df,["eid"],"inner")
emp_city_inner_df.show()

### Full Join 
combining both tables all matched and unmatched records (both left and right join)

In [0]:
emp_data_df.show()
city_data_df.show()

emp_city_full_df=emp_data_df.join(city_data_df,["eid"],"full")
emp_city_full_df.show()

### what happens if we have Null values /duplicate values in Join key

In [0]:
#null values
emp_data=[(100,"A0",25),(101,"A1",36),(102,"A2",26),(103,"A3",32),(104,"A4",12),(105,"A5",54),(106,"A6",32),(None,"A7",32)]

city_data=[(100,"chn"),(104,"blr"),(105,"hyd"),(106,"mum"),(107,"hyd"),(110,"del"),(None,"pune")]

emp_data_df=spark.createDataFrame(emp_data,["eid","ename","age"])
city_data_df=spark.createDataFrame(city_data,["eid","city"])

emp_data_df.show()
city_data_df.show()


#default is inner join
#inner join - will not show any null records
#right join - if right table have null records then it will show
#left join - if left table have null records then it will show
#full join - it will show all the null records

#Handling Null values in join
emp_city_null_df=emp_data_df.join(city_data_df,emp_data_df.eid==city_data_df.eid)
emp_city_null_df.show()   


#show the left table eid null record and return the df based on left join
emp_city_null_df=emp_data_df.join(city_data_df,emp_data_df.eid==city_data_df.eid,"right")
emp_city_null_df.show()


#none of the eid null records wil return
emp_city_null_df=emp_data_df.join(city_data_df,emp_data_df.eid==city_data_df.eid,"inner")
emp_city_null_df.show()

#it will show all eid null records from both the table
emp_city_null_df=emp_data_df.join(city_data_df,emp_data_df.eid==city_data_df.eid,"full")
emp_city_null_df.show()



### Null Safe Join

In [0]:
#Null safe join - when you want to compare null==null
# in sql, <=>
emp_city_null_df=emp_data_df.join(city_data_df,emp_data_df.eid.eqNullSafe(city_data_df.eid),"inner")
emp_city_null_df.show()

### Duplicate records - Cross join

In [0]:
#Duplicate records - for duplicated id, its doing the cross join.

emp_data=[(100,"A0",25),(101,"A1",36),(102,"A2",26),(103,"A3",32),(104,"A4",12),(105,"A5",54),(106,"A6",32),(105,"A7",32)]

city_data=[(100,"chn"),(104,"blr"),(105,"hyd"),(106,"mum"),(107,"hyd"),(110,"del"),(104,"pune")]

emp_data_df=spark.createDataFrame(emp_data,["eid","ename","age"])
city_data_df=spark.createDataFrame(city_data,["eid","city"])

emp_data_df.show()
city_data_df.show()

#Handling Duplicate records in join
#default is inner join

emp_city_inner_df=emp_data_df.join(city_data_df,"eid","inner")
emp_city_inner_df.show()
emp_city_left_df=emp_data_df.join(city_data_df,emp_data_df.eid==city_data_df.eid,"left")
emp_city_left_df.show()

emp_city_inner_df=emp_data_df.join(city_data_df,emp_data_df.eid==city_data_df.eid,"inner")
emp_city_inner_df.show()

### Cross Join - (M*N)

M- records from table A

N- records from table B

In [0]:
emp_data=[(100,"A0",25),(101,"A1",36)]
city_data=[(100,"chn"),(104,"blr")]

emp_data_df=spark.createDataFrame(emp_data,["eid","ename","age"])
city_data_df=spark.createDataFrame(city_data,["eid","city"])

emp_data_df.show()  
city_data_df.show()

#Cross join define in 2 ways.
# 1. join without any condition, then its a cross join.
# 2. join with cross join
emp_city_df=emp_data_df.join(city_data_df)
emp_city_df.show()

emp_city_df=emp_data_df.crossJoin(city_data_df)
emp_city_df.show()


#spark defaults
#join without on condition is a cross join
#join with condition and without how is inner join

###  left semi - similar to exists
based on the matching condition , it will return only left side table
whereas in  inner join, it will return both left and right side columns of matching records

customer : cid ,cname

trans : tid,cid,sal_amt,sal_dt..

**need a customer list , who made purchases last 2 months - left semi**

**need a customer id with total_trans, tans_amount ,from last 2 months data - inner**

### left anti - not exists

**need a customer list , who didn't made any purchases last 2 months - left anti**



### left anti - similar to not exists - opposite of left semi

based on matching condition, it will return only left side table

In [0]:
emp_data=[(100,"A0",25),(101,"A1",36),(102,"A2",26),(103,"A3",32),(104,"A4",12),(105,"A5",54),(106,"A6",32),(None,"A7",32)]

city_data=[(100,"chn"),(100,"chn2"),(105,"hyd"),(106,"mum"),(107,"hyd"),(110,"del"),(None,"pune")]

emp_data_df=spark.createDataFrame(emp_data,["eid","ename","age"])
city_data_df=spark.createDataFrame(city_data,["eid","city"])

emp_data_df.show()
city_data_df.show()


emp_city_leftsemi_df=emp_data_df.join(city_data_df,emp_data_df.eid==city_data_df.eid,"left")
emp_city_leftsemi_df.show()

#left semi - only matching records from left side
# employees who have city details
emp_city_leftsemi_df=emp_data_df.join(city_data_df,emp_data_df.eid==city_data_df.eid,"leftsemi")
display(emp_city_leftsemi_df)

#only non matching records from left side - not exists
# employees who dont have city details
emp_city_leftanti_df=emp_data_df.join(city_data_df,emp_data_df.eid==city_data_df.eid,"leftanti")
display(emp_city_leftanti_df)

### Spark internals
 - when shuffle happens this magic happen in spark internally , (performance side)

Broadcast join

Shuffle join

sort merge join

AQE - adoptive Query execution


# Windowing

**window - set of related rows / grouped rows**

**similar to group by , but difference is group by collapse the record**

Window function - will put all related records in one frame

**diffrent types of windowing functions**

- Ranking - row_number,rank,dense_rank ...

- Analytical - lead , lag , first , last ....

- Aggregate - sum , avg , min ...



In [0]:
data =[ ('Lisa', 'Sales', 10000, 35),
          ('Evan', 'Sales', 32000, 38),
          ('Fred', 'Engineering', 21000, 28),
          ('Alex', 'Sales', 30000, 33),
          ('Tom', 'Engineering', 23000, 33),
          ('Jane', 'Marketing', 29000, 28),
          ('Jeff', 'Marketing', 35000, 38),
          ('Paul', 'Engineering', 29000, 23),
          ('Chloe', 'Engineering', 23000, 25)]
        
data_df=spark.createDataFrame(data,["name", "dept", "salary", "age"])
data_df.show()

**there is no id column in the dataset **

** source sending (natural key ) column which is not unique, so i need to generate one sequence number (surrgate key - sk ) for my table.**

 


# monotonically increasing id

**sequence number i want to generate in spark without Windowing function then use monotonically increasing id**

if we use this we will get a unique and sequence record, but only for smaller records (single partition).

if the records is huge and distributed among multiple partitions, monotonically increasing id will give unique number but not in the sequence number.




In [0]:
from pyspark.sql.functions import monotonically_increasing_id

data_df1=data_df.withColumn("ID",monotonically_increasing_id()).show()

data_df2=data_df.repartition(3).withColumn("ID",monotonically_increasing_id()).show()

# if data is in same partition , we will get unique sequence numbers 
# if the data in multiple partitions , we will get unique but numbers are not sequence 


## Window + Ranking Functions 

provides rank to dataset if it is a  different partitions then start from 1

- row_number - provides the sequence from 1  and generate the sequence for each partition
- rank - if there are duplicates , they will get same rank and next rank will be skipped
- dense_rank - if there are duplicates , they will get same rank and next rank will be incremented

In [0]:
#row_number
#syntax: function().over(window)
#window - partition_by + orderby
#rownumber().over(partition by dept order by salary)


from pyspark.sql.functions import row_number,rank,dense_rank
from pyspark.sql.window import Window
from pyspark.sql.functions import desc

windowSpec=Window.partitionBy("dept").orderBy("salary")
ranking_df=data_df.withColumn("row_number",row_number().over(windowSpec)).show()

ranking_df=data_df.withColumn("row_number",row_number().over(windowSpec)).withColumn("rank",rank().over(windowSpec)).withColumn("dense_rank",dense_rank().over(windowSpec))

ranking_df.show()

# show who is getting 2nd lowest salary
ranking_df.filter(ranking_df.dense_rank==2).show()

#show who is getting 2nd highest salary
windowSpec=Window.partitionBy("dept").orderBy(desc("salary"))

ranking_df1=data_df.withColumn("dense_rank",dense_rank().over(windowSpec))
ranking_df1.show()

# show who is getting 2nd highest salary
ranking_df1.filter(ranking_df1.dense_rank==2).show()





In [0]:
# global rank 

# row_number -> seq number / rank for each partition

from pyspark.sql.functions import row_number,rank,dense_rank
from pyspark.sql.window import Window
from pyspark.sql.functions import desc


data_df.show()

rank_df=data_df.withColumn("desc_rank",row_number().over(Window.orderBy(desc("age"))))
rank_df.show()

rank_df=data_df.withColumn("desc_rank",row_number().over(Window.partitionBy("dept").orderBy(desc("age"))))
rank_df.show()

rank_df1=data_df.select("*",row_number().over(Window.orderBy("age")).alias("asc_rank")).show()

rank_df1=data_df.select("*",row_number().over(Window.partitionBy("dept").orderBy("age")).alias("asc_rank")).show()


In [0]:
# each department get top 2 salary - (if ties also bring all records)

from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window

windowSpec=Window.partitionBy("dept").orderBy(desc("salary"))

# department wise 2nd maximum salary 
top2_salary_df=data_df.withColumn("salary_rank",dense_rank().over(windowSpec))
top2_salary_df.show()
top2_salary_df.filter(top2_salary_df.salary_rank==2).show()


# department wise 2nd minimum salary 
min2_salary_df=data_df.withColumn("min2_salary",dense_rank().over(Window.partitionBy("dept").orderBy("salary"))).filter("min2_salary==2").show()


In [0]:
# rank based on sal desc and if sal ties consider age also , least age give priority 

windowSpec=Window.partitionBy("dept").orderBy(desc("salary"),"age")

df1=data_df.withColumn("desc_rank",dense_rank().over(windowSpec))
df1.show()


In [0]:
#save as table

df1.write.mode("overwrite").saveAsTable("izwd37dev.wd37db.tbl_sal")

In [0]:
%sql
select * from izwd37dev.wd37db.tbl_sal

### Top N analysis using standard SQL

In [0]:
%sql

-- global ranking based on sal
select *,row_number() over(order by salary desc) as row_number from izwd37dev.wd37db.tbl_sal;


-- global ranking based on age and sal
select *,row_number() over(order by age,salary desc)as row_number from izwd37dev.wd37db.tbl_sal;

-- global ranking based on dept and sal
select *,row_number() over(partition by dept order by salary desc) as rnk from izwd37dev.wd37db.tbl_sal;

### window - Analytical functions
- lag
- lead
- first value
- last_value

In [0]:
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/custs_header1",header=True)
cust_df.show(10)


In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/csv/txns_2025.txt

In [0]:
trans_schema="trans_id long,trans_date string,cust_id long,trans_amt float,category string,accessories string,city string,state string,mode string"
trans_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/csv/txns_2025.txt",header=False,schema=trans_schema)
trans_df.show(3,False)

# EDA - duplicate / nullability check
trans_df.count()   # 95904

trans_df.distinct().count()  # 95904

trans_df.printSchema()

#changing trans_date string to dateformat
from pyspark.sql.functions import to_date,col
trans_df=trans_df.withColumn("trans_date",to_date(col("trans_date"),"MM-dd-yyyy"))
trans_df.show(3)
trans_df.printSchema()

In [0]:
#joining transaction df and cust_header df

joined_df=trans_df.join(cust_df,trans_df.cust_id==cust_df.custid,"inner")
joined_df.show(5)
joined_df.count()  #95892


# retrieve cust_id, txnid,fname,txndt,amt
from pyspark.sql.functions import to_date,col,concat,lit
trans_cust_df=joined_df.select(joined_df.cust_id,joined_df.trans_id,concat(col("fname"),lit(" "),col("lname")).alias("FullName"),joined_df.trans_date,joined_df.trans_amt)
trans_cust_df.show(5)

trans_cust_filter_df=trans_cust_df.filter("custid=4000001").orderBy("trans_date")
trans_cust_filter_df.show()



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number



w_spec=Window.partitionBy("cust_id").orderBy("trans_date")

trans_cust_df1=trans_cust_filter_df.withColumn("rnk",row_number().over(w_spec))
trans_cust_df1.show()


In [0]:
# lead and lag 

# lead(col,offset,default=None)

# lag(col,offset,default=None)
from pyspark.sql.window import Window
from pyspark.sql.functions import lead,lag
window_spec = Window.partitionBy("cust_id").orderBy("trans_date") 
trans_cust_df2=trans_cust_df1.withColumn("next_trans",lead("trans_amt",1).over(window_spec))

trans_cust_df2.show()

trans_cust_df3=trans_cust_df2.withColumn("prev_trans",lag("trans_amt",2).over(window_spec))
trans_cust_df3.show()

### Grouping and Aggregate - sum , avg , min ...

In [0]:
data =[ ('Lisa', 'Sales', 10000, 35),
          ('Evan', 'Sales', 32000, 38),
          ('Fred', 'Engineering', 21000, 28),
          ('Alex', 'Sales', 30000, 33),
          ('Tom', 'Engineering', 23000, 33),
          ('Jane', 'Marketing', 29000, 28),
          ('Jeff', 'Marketing', 35000, 38),
          ('Paul', 'Engineering', 29000, 23),
          ('Chloe', 'Engineering', 23000, 25)]

data_df = spark.createDataFrame(data, ['name', 'dept', 'salary', 'age'])
data_df.show()

In [0]:
# dept wise total salary
from pyspark.sql.functions import sum, avg, min, max

data_df.groupby("dept").agg(sum("salary")).show()
data_df.groupby("dept").agg(sum("salary"),avg("salary"),min("salary"),max("salary")).show()




In [0]:
# identify salary of person based on dept avg salary higher , lower 

from pyspark.sql.window import Window
from pyspark.sql.functions import when
window_spec=Window.partitionBy("dept")

agg_df=data_df.withColumn("min_salary",min("salary").over(window_spec)).withColumn("max_salary",max("salary").over(window_spec)).withColumn("avg_sal",avg("salary").over(window_spec)).withColumn("sum_sal",sum("salary").over(window_spec))
agg_df.show()

final_df=agg_df.select("name","dept","salary","avg_sal").withColumn("salary_status",when(col("salary")>col("avg_sal"),"higher").when(col("salary")<col("avg_sal"),"lower").otherwise("equal"))
final_df.show()

In [0]:
trans_cust_df.show(10)
from pyspark.sql.functions import year,month
grouped_df=trans_cust_df.groupBy(year("trans_date"),month("trans_date")).count().orderBy(month("trans_date"))
grouped_df.show()


In [0]:
# when you want to find cummulative salary of each department, use sum

data_df.show()
from pyspark.sql.window import Window
from pyspark.sql.functions import sum

window_spec=Window.partitionBy("dept").orderBy("age")
data_df.withColumn("sum_salary",sum("salary").over(window_spec)).show()

In [0]:
# when you want to find cummulative amount of a person purchased 

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number



w_spec=Window.partitionBy("cust_id").orderBy("trans_date")

trans_cust_df1=trans_cust_filter_df.withColumn("rnk",row_number().over(w_spec))
trans_cust_df1.show()

cummulative_df=trans_cust_df1.withColumn("cum_trans",sum("trans_amt").over(w_spec))
cummulative_df.show()


# Special functions - Cube,Pivot,rollup


## rollup

is similar to groupBy function but the difference is its going in hierrarchial order

rollup - hierachical combination

(dept,age) --> sum()

(dept) ->sum

()-sum

In [0]:
data_df.show()

from pyspark.sql.functions import desc
grouped_df=data_df.groupBy("dept","age").agg(sum("salary")).orderBy("dept")
grouped_df.show()

rollup_df=data_df.rollup("dept","age").agg(sum("salary")).orderBy("dept")
rollup_df.show()





## Cube

cube - every combination of dept and age

dept+age -> sum
dept-> sum
age-> sum
sum

a,b,c -> sum
a,b
a,c
b,c
a,
b
c
#

In [0]:
# cube - every combination of dept and age

data_df.show()
data_df.cube("dept","age").sum("salary").orderBy("dept").show()


## Pivot 
Pivot are nothing but changing rows to columns

In [0]:

trans_cust_df.show(5)
joined_df.show(5)

trans_cust_pivot_df=joined_df.select(cust_df.custid,"trans_id","fname","trans_date","trans_amt","state","city","mode")
trans_cust_pivot_df.show(5)

trans_cust_pivot_df.filter("state in ('Alabama','California')").groupBy("state","city").pivot("mode").sum("trans_amt").orderBy("city").show()


# set Operations 

### union , intersection , subtract 


pyspark DSL -> union = union all -> combining DF (similar)

## Union

In [0]:
# union or unionByName or union All - it will not remove any duplicates in DSL

data1=[(100,"A1",34),(101,"A2",31),(102,"A3",43)]
data2=[(100,"A1",34),(101,"B2",11),(502,"B3",42)]

data1=spark.createDataFrame(data1,['id','name','age'])
data2=spark.createDataFrame(data2,['id','name','age'])

data1.union(data2).show()
data1.unionByName(data2).show()
data1.unionAll(data2).show()

## Intersection

In [0]:
#intersection - shows only the matching records in data1

data1.intersect(data2).show()


## Substract 



In [0]:
# shows only non matching records in data1
data1.subtract(data2).show()